# Word2Vec

### Stopword Removal
- Stopwords are common words such as **"the"**, **"is"**, **"and"**, **"of"**, etc.
- These words help in sentence formation but usually do not contribute much to the meaning of the sentence.
- Removing stopwords reduces noise and helps the model focus on important words.

### Implementation
- Stopword removal can be performed using the **Scikit-learn** library.
- We use the built-in English stopword list provided by Scikit-learn.

# Step 1: Import dependencies

In [7]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')

# Step 2: Count Vectorizer, to convert text into numberical vectors
`max_features=5000`

- Keep only the 5000 most frequent words in the dataset.
- This reduces dimensionality and memory usage.

In [11]:
# convert tags into vectors
movies = pd.read_csv("/content/movies.csv")
vectors = cv.fit_transform(movies['tags']).toarray()
vectors

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [12]:
# view the vocabulary
print(cv.get_feature_names_out())

['000' '007' '10' ... 'zone' 'zoo' 'zooeydeschanel']


In [13]:
# vocabulary size
len(cv.get_feature_names_out())

5000

# Step 3: Stemming

In [14]:
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [15]:
print(ps.stem('action'))
print(ps.stem('actions'))

action
action


In [16]:
print(ps.stem('danced'))
print(ps.stem('dance'))
print(ps.stem('dancing'))

danc
danc
danc


In [17]:
print(ps.stem('love'))
print(ps.stem('loving'))

love
love


In [18]:
def stem(text):
  y = []
  for i in text.split():
    y.append(ps.stem(i))
  return " ".join(y)

In [20]:
movies['tags'] = movies['tags'].apply(stem)

In [21]:
movies['tags']

,tags
0,"in the 22nd century, a parapleg marin is dispa..."
1,"captain barbossa, long believ to be dead, ha c..."
2,a cryptic messag from bond’ past send him on a...
3,follow the death of district attorney harvey d...
4,"john carter is a war-weary, former militari ca..."
...,...
4801,el mariachi just want to play hi guitar and ca...
4802,a newlyw couple' honeymoon is upend by the arr...
4803,"""signed, sealed, delivered"" introduc a dedic q..."
4804,when ambiti new york attorney sam is sent to s...


# Step 4: Similarity (Cosine Similarity)

# Why Cosine Distance over Euclidian Distance

- We will calculate cosine distance
- Euclidian distance will calculate tip to tip distance
- Cosine will calculate degree beteen them
- If high dimension, euclidian distance is not a reliable measure.
- So, we will Use cosine similarity

$$
\text{Distance} = \frac{1}{\text{Similarity}}
$$

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
similarity = cosine_similarity(vectors)
similarity

array([[1.        , 0.08740748, 0.05827165, ..., 0.02418254, 0.02564946,
        0.        ],
       [0.08740748, 1.        , 0.06451613, ..., 0.02677398, 0.        ,
        0.        ],
       [0.05827165, 0.06451613, 1.        , ..., 0.02677398, 0.        ,
        0.        ],
       ...,
       [0.02418254, 0.02677398, 0.02677398, ..., 1.        , 0.07071068,
        0.04836508],
       [0.02564946, 0.        , 0.        , ..., 0.07071068, 1.        ,
        0.05129892],
       [0.        , 0.        , 0.        , ..., 0.04836508, 0.05129892,
        1.        ]])

In [24]:
similarity.shape

(4806, 4806)

> **Note:** Total 4806 movies, each moview distance with all other 4806 movies

> it is a array of array

In [25]:
similarity[[1]]

array([[0.08740748, 1.        , 0.06451613, ..., 0.02677398, 0.        ,
        0.        ]])

> **Note:** We can see, 1 , it means, for 1st movie the similarity with itself is 1 or 100

> So, diagonal will always be `1`

# Step 5: Return top 5 similar movies

how to do it?
- Create a index for all movies
- Fetch the similarity(vactor)
- sort it
- return top 5 movies by the indexs

In [27]:
# Index
print(movies[movies['title'] == 'Avatar'].index[0])

0


In [28]:
print(movies[movies['title'] == 'Batman Begins'].index[0])


119


In [29]:
# Problem, after sorting we loose index position
print(sorted(similarity[0],reverse=True))

[np.float64(1.0000000000000002), np.float64(0.25038669783359574), np.float64(0.24779731389167606), np.float64(0.24283093212859141), np.float64(0.2409900932515112), np.float64(0.23939494881986934), np.float64(0.233785950059759), np.float64(0.23174488732966075), np.float64(0.23084512921915967), np.float64(0.23084512921915967), np.float64(0.2294157338705618), np.float64(0.21677749238103003), np.float64(0.21629522817435), np.float64(0.21526419295572297), np.float64(0.21486752129677), np.float64(0.21296183592613546), np.float64(0.21239769762143662), np.float64(0.21239769762143662), np.float64(0.2108663315950723), np.float64(0.20857039859669466), np.float64(0.20770324619863198), np.float64(0.20751433915982237), np.float64(0.20395079136182276), np.float64(0.20395079136182276), np.float64(0.2029530274475215), np.float64(0.2029530274475215), np.float64(0.2024645717996314), np.float64(0.19767387315371682), np.float64(0.19466570535691505), np.float64(0.19194297398747862), np.float64(0.19117977822

In [30]:
# Solve using enumerate function
list(enumerate(similarity[0]))

[(0, np.float64(1.0000000000000002)),
 (1, np.float64(0.08740748201220976)),
 (2, np.float64(0.0582716546748065)),
 (3, np.float64(0.03823595564509363)),
 (4, np.float64(0.177343107178349)),
 (5, np.float64(0.11357771260606365)),
 (6, np.float64(0.019389168358237032)),
 (7, np.float64(0.1692777916923361)),
 (8, np.float64(0.06131393394849658)),
 (9, np.float64(0.07336739820667779)),
 (10, np.float64(0.11295649894498103)),
 (11, np.float64(0.07792865001991967)),
 (12, np.float64(0.09365858115816941)),
 (13, np.float64(0.04588314677411236)),
 (14, np.float64(0.10968169942141635)),
 (15, np.float64(0.04947706959952935)),
 (16, np.float64(0.07894736842105264)),
 (17, np.float64(0.1442149876003076)),
 (18, np.float64(0.10743376064838502)),
 (19, np.float64(0.08240856434303293)),
 (20, np.float64(0.055824219567359015)),
 (21, np.float64(0.08885233166386385)),
 (22, np.float64(0.0662266178532522)),
 (23, np.float64(0.09365858115816941)),
 (24, np.float64(0.0533380747062665)),
 (25, np.float64

In [31]:
sorted(list(enumerate(similarity[0])),  reverse=True,  key=lambda x:x[1])[1:6]

# lambda function to tell, sorting need to be done by [1]-->vector not [0]--> enumerate value

[(539, np.float64(0.25038669783359574)),
 (1192, np.float64(0.24779731389167606)),
 (507, np.float64(0.24283093212859141)),
 (260, np.float64(0.2409900932515112)),
 (1214, np.float64(0.23939494881986934))]

In [33]:
# Print movie name, using enumerates
movies.iloc[1214].title

'Aliens vs Predator: Requiem'

In [34]:
def recommend(movie):
  #Index
  movie_index = movies[movies['title'] == movie].index[0]
  distances = similarity[movie_index]

  # Sorting and top 5
  movies_list = sorted(list(enumerate(distances)),  reverse=True,  key=lambda x:x[1])[1:6]

  # Print movies
  for i in movies_list:
    print(movies.iloc[i[0]].title)
  return

In [35]:
recommend('Avatar')

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


In [36]:
recommend('Batman Begins')

The Dark Knight
The Dark Knight Rises
Batman
Batman
Batman & Robin


# Step 6: Save using Pickle Library

In [37]:
import pickle
pickle.dump(movies, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
